In [1]:

# %%
# ==============================================
# Week 4 Section 2 — Retraining & Validation (Business)
# Single-cell; profile-aware; reads best params from Section 1
# ==============================================

# Config
PROFILE = "dev"   # dev | preprod | final
SECTION_ID = "Wk04_Section2"

CFG = {
    "dev":     dict(max_days=90,   zones=1, test_days=7),
    "preprod": dict(max_days=365,  zones=3, test_days=28),
    "final":   dict(max_days=None, zones=3, test_days=28),
}
cfg = CFG.get(PROFILE, CFG["dev"])

from pathlib import Path
BASE_DIR = Path.cwd()
if BASE_DIR.name.lower() == "code":
    BASE_DIR = BASE_DIR.parent
RESULTS_DIR = BASE_DIR / "results" / f"{SECTION_ID}_{PROFILE}"
REPORTS_DIR = RESULTS_DIR / "reports"
PLOTS_DIR   = RESULTS_DIR / "plots"
for d in (REPORTS_DIR, PLOTS_DIR): d.mkdir(parents=True, exist_ok=True)

DATA_PATH   = BASE_DIR / "data" / "Tetuan City power consumption.csv"
S1_BEST     = BASE_DIR / "results" / f"Wk04_Section1_{PROFILE}" / "reports" / "best_params.json"
CONSOLIDATED = BASE_DIR / "SDS-CP036-powercast_Wk04_Report_Business.md"

print(f"[BOOT] BASE_DIR={BASE_DIR} PROFILE={PROFILE} -> {RESULTS_DIR}")


# Common helpers
from pathlib import Path
import os, re, math, warnings, json
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Cap threads to keep runtime light
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Optional libraries
_HAVE_SM = False
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    _HAVE_SM = True
except Exception:
    _HAVE_SM = False

_HAVE_FBPROPHET = False
try:
    try:
        from prophet import Prophet
        _HAVE_FBPROPHET = True
    except Exception:
        from fbprophet import Prophet
        _HAVE_FBPROPHET = True
except Exception:
    _HAVE_FBPROPHET = False

_HAVE_XGB = False
try:
    from xgboost import XGBRegressor as _XGB
    def XGBRegressor(*args, **kwargs):
        kwargs.setdefault("n_estimators", 200)
        kwargs.setdefault("max_depth", 4)
        kwargs.setdefault("learning_rate", 0.1)
        kwargs.setdefault("subsample", 0.9)
        kwargs.setdefault("colsample_bytree", 0.9)
        kwargs.setdefault("tree_method", "hist")
        kwargs.setdefault("n_jobs", 1)
        return _XGB(*args, **kwargs)
    _HAVE_XGB = True
except Exception:
    _HAVE_XGB = False

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"\s+", " ", c.strip()) for c in df.columns]
    return df

def infer_step_and_freq(idx: pd.DatetimeIndex):
    try:
        freq = pd.infer_freq(idx[:1000])
    except Exception:
        freq = None
    if freq:
        step = pd.to_timedelta(pd.tseries.frequencies.to_offset(freq))
        return step, freq
    diffs = pd.Series(idx).diff().dropna()
    step = diffs.mode().iloc[0] if not diffs.empty else pd.Timedelta(hours=1)
    total_sec = int(step.total_seconds())
    if total_sec % 3600 == 0:
        freq = f"{total_sec // 3600}H"
    elif total_sec % 60 == 0:
        freq = f"{total_sec // 60}T"
    else:
        freq = f"{total_sec}S"
    return step, freq

def limit_last_days(df: pd.DataFrame, date_col="DateTime", max_days=None) -> pd.DataFrame:
    if not max_days:
        return df
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    cutoff = df[date_col].max() - pd.Timedelta(days=max_days)
    return df[df[date_col] >= cutoff]

def pick_zone_cols(df: pd.DataFrame, k: int):
    cols = [c for c in df.columns if re.match(r"Zone\s+\d+\s+Power\s+Consumption", c)]
    cols = sorted(cols, key=lambda s: int(re.search(r"\d+", s).group()))
    return cols[:k] if k else cols

def make_time_splits(df: pd.DataFrame, test_days: int, date_col="DateTime", steps_per_day: int = 24):
    df = df.sort_values(date_col)
    test_len = int(test_days * steps_per_day)
    if test_len >= len(df):
        test_len = max(1, len(df)//10)
    test = df.iloc[-test_len:].copy()
    train_val = df.iloc[:-test_len].copy()
    if len(train_val) > 100:
        cut = int(len(train_val) * 0.85)
        train, val = train_val.iloc[:cut].copy(), train_val.iloc[cut:].copy()
    else:
        train, val = train_val.copy(), pd.DataFrame(columns=train_val.columns)
    return train, val, test

def metrics_series(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    denom = np.maximum(1e-8, np.where(y_true==0, 1e-8, y_true))
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)
    return dict(rmse=rmse, mae=mae, mape=mape)

def save_plot_ts(ts_df: pd.DataFrame, cols, title, path: Path):
    plt.figure(figsize=(10,4))
    for c in cols:
        plt.plot(ts_df.index, ts_df[c], label=c)
    plt.title(title); plt.xlabel("Time"); plt.ylabel("Power")
    plt.legend(loc="best"); plt.tight_layout()
    plt.savefig(path, bbox_inches="tight"); plt.close("all")

def rel(p: Path, base: Path) -> str:
    try: return p.relative_to(base).as_posix()
    except Exception: return p.as_posix()

def write_text(path: Path, text: str):
    path.write_text(text, encoding="utf-8")
    print("Wrote:", rel(path, Path.cwd()))


if not DATA_PATH.exists():
    write_text(REPORTS_DIR / "SDS-CP036-powercast_Wk04_Section2_Business_Report.md",
               "# SDS-CP036-powercast – Week 4 Section 2: Retraining & Validation\n\n"
               f"Profile: `{PROFILE}`\n\n**Error:** data file not found: {rel(DATA_PATH, BASE_DIR)}")
    raise SystemExit(0)

df = pd.read_csv(DATA_PATH)
df = normalize_columns(df)
if "DateTime" not in df.columns:
    alts = [c for c in df.columns if c.lower()=="datetime"]
    if alts: df = df.rename(columns={alts[0]:"DateTime"})
zones = pick_zone_cols(df, cfg["zones"])
if not zones: raise RuntimeError("No zone columns found.")

work = df["DateTime"].to_frame().join(df[zones])
work["DateTime"] = pd.to_datetime(work["DateTime"])
work = work.sort_values("DateTime").set_index("DateTime")

step_td, freq_str = infer_step_and_freq(work.index)
steps_per_hour = max(1, int(round(pd.Timedelta(hours=1) / step_td)))
steps_per_day  = int(24 * steps_per_hour)

train, val, test = make_time_splits(work.reset_index(), cfg["test_days"], "DateTime", steps_per_day=steps_per_day)
train = train.set_index("DateTime"); val = val.set_index("DateTime"); test = test.set_index("DateTime")

# Load best params
best_by_zone = {}
if S1_BEST.exists():
    best_by_zone = json.loads(S1_BEST.read_text(encoding="utf-8"))
else:
    print("[WARN] best_params.json not found, will fallback to SARIMAX default.")

def fit_predict_sarimax(series_native, horizon_steps):
    ser_hr = pd.to_numeric(series_native, errors="coerce").ffill().resample("1H").mean().asfreq("H").ffill()
    if len(ser_hr) > 2400: ser_hr = ser_hr.iloc[-2400:]
    model = SARIMAX(ser_hr, order=(1,0,1), seasonal_order=(0,1,1,24),
                    enforce_stationarity=False, enforce_invertibility=False,
                    simple_differencing=True, trend=None, concentrate_scale=True)
    res = model.fit(method="lbfgs", maxiter=40, disp=False)
    h = int(math.ceil(horizon_steps / steps_per_hour))
    fc = res.get_forecast(steps=h).predicted_mean.values.astype(float)
    idx_hr = pd.date_range(ser_hr.index[-1]+pd.Timedelta(hours=1), periods=h, freq="H")
    return pd.Series(fc, index=idx_hr)

def make_lag_supervised(y: pd.Series, max_lag=24):
    y = pd.to_numeric(y, errors="coerce").ffill()
    df_sup = pd.DataFrame({"y": y})
    for L in range(1, max_lag+1):
        df_sup[f"lag_{L}"] = y.shift(L)
    return df_sup.dropna()

rows = []
for zone in zones:
    print(f"[ZONE] {zone}")
    series_full = pd.concat([train[zone], val[zone]])
    choice = best_by_zone.get(zone, {})
    model_name = choice.get("model", "SARIMAX")
    params = choice.get("params")

    if model_name == "XGB" and _HAVE_XGB:
        par = json.loads(params) if isinstance(params, str) else (params or {})
        sup = make_lag_supervised(series_full, 24)
        X, y = sup.drop(columns=["y"]).values, sup["y"].values
        model = XGBRegressor(**par)
        model.fit(X, y, verbose=False)
        hist = list(series_full.values)
        preds = []
        for _ in range(len(test)):
            feats = [hist[-k] for k in range(1,25)]
            preds.append(float(model.predict(np.array(feats).reshape(1,-1))))
            hist.append(preds[-1])
        y_pred = np.array(preds)
        comp = pd.DataFrame({"actual": test[zone], "pred": y_pred}, index=test.index)
        m = metrics_series(test[zone].values, y_pred)
        rows.append({"zone":zone,"model":"XGB (tuned)","rmse":m["rmse"],"mae":m["mae"],"mape":m["mape"]})
        save_plot_ts(comp, ["actual","pred"], f"{zone} — Tuned XGB vs Actual", PLOTS_DIR / f"{zone.replace(' ','_')}_xgb_tuned_vs_actual.png")
    else:
        y_hr = fit_predict_sarimax(series_full, len(test))
        native_hours = pd.to_datetime(test.index.floor("H"))
        y_pred = y_hr.reindex(native_hours, method="ffill").values[:len(test)]
        comp = pd.DataFrame({"actual": test[zone], "pred": y_pred}, index=test.index)
        m = metrics_series(test[zone].values, y_pred)
        rows.append({"zone":zone,"model":"SARIMAX (tuned/candidate)","rmse":m["rmse"],"mae":m["mae"],"mape":m["mape"]})
        save_plot_ts(comp, ["actual","pred"], f"{zone} — SARIMAX vs Actual", PLOTS_DIR / f"{zone.replace(' ','_')}_sarimax_vs_actual.png")

metrics_csv = REPORTS_DIR / "retrain_test_metrics.csv"
pd.DataFrame(rows).to_csv(metrics_csv, index=False)

# Report
s2_md = REPORTS_DIR / "SDS-CP036-powercast_Wk04_Section2_Business_Report.md"
write_text(s2_md, f"""
# SDS-CP036-powercast – Week 4 Section 2: Retraining & Validation

Profile: `{PROFILE}`

### Key Questions Answered
**Q: How did you retrain your best models on the full training data after tuning?**  
**A:** We **combined train+validation** and re-fit the chosen model per zone (from Section 1).

**Q: What steps did you take to validate model performance on the test set?**  
**A:** We kept the **chronological test slice untouched** during tuning, then forecasted its horizon and captured **RMSE/MAE/MAPE**.

**Q: Did you observe changes after retraining? How did you interpret them?**  
**A:** Minor metric shifts are expected (more data → steadier estimates). If accuracy improved, the model benefited from more context; if it dipped slightly, that suggests **variance vs. bias** trade-offs and highlights where to adjust features or parameters.

### Artifacts
- Test metrics: `{rel(metrics_csv, BASE_DIR)}`  
- Plots (actual vs predicted) in: `{rel(PLOTS_DIR, BASE_DIR)}/`

### Business Value (Executive View)
- **Trustworthy test numbers** (tuning never saw the test data).  
- **Closer-to-production fit** by retraining on the full learning window.
""")

# Upsert consolidated
def upsert_section(consolidated_path: Path, section_idx: int, title: str, new_block: str):
    if not consolidated_path.exists():
        consolidated_path.write_text("# SDS-CP036-powercast – Week 4 (Business Report)\n\n", encoding="utf-8")
    text = consolidated_path.read_text(encoding="utf-8")
    hdr_any = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*4\s*Section\s*[1-5]\s*:\s*.*$")
    hdr_me = re.compile(fr"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*4\s*Section\s*{section_idx}\s*:\s*{title}\s*$")
    # remove existing block
    m = hdr_me.search(text)
    if m:
        start = m.start()
        m2 = hdr_any.search(text, pos=m.end())
        end = m2.start() if m2 else len(text)
        text = (text[:start] + text[end:]).strip()
    # append
    text = text.rstrip() + "\n\n" + new_block.strip() + "\n"
    consolidated_path.write_text(text, encoding="utf-8")

upsert_section(CONSOLIDATED, 2, "Retraining \& Validation", s2_md.read_text(encoding="utf-8"))
print("✅ Upserted Section 2 into consolidated:", rel(CONSOLIDATED, BASE_DIR))


[BOOT] BASE_DIR=/home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast PROFILE=dev -> /home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast/results/Wk04_Section2_dev
[ZONE] Zone 1 Power Consumption
Wrote: /home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast/results/Wk04_Section2_dev/reports/SDS-CP036-powercast_Wk04_Section2_Business_Report.md
✅ Upserted Section 2 into consolidated: SDS-CP036-powercast_Wk04_Report_Business.md
